In [1]:
import os
import sys
import pandas as pd
import numpy as np
os.getcwd()

'C:\\Users\\TWH'

In [2]:
# Auto-install MLflow if missing
try:
    import mlflow
    import mlflow.sklearn
    import mlflow.tensorflow
except ImportError:
    !{sys.executable} -m pip install mlflow
    import mlflow
    import mlflow.sklearn
    import mlflow.tensorflow
    
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    mean_absolute_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

In [3]:
class TravelTimingRecommender:
    """Travel Timing Recommendation Engine for Interstate Corridor Travel.

    Evaluates traffic volume and risk metrics across day types, weather, and
    hourly windows to deliver optimal departure times.
    """

    def __init__(self, dataset_path="data/processed/cleaned_traffic_features.csv"):
        """Initialize recommender with data source or synthetic fallbacks."""
        try:
            self.df = pd.read_csv(dataset_path)
        except FileNotFoundError:
            # Generate representative fallback corridor data if file path does not exist
            self.df = self._generate_synthetic_corridor_data()

        self._build_lookup_profile()

    def _generate_synthetic_corridor_data(self):
        """Generates representative hourly corridor traffic data for fallback execution."""
        np.random.seed(42)
        records = []
        for hour in range(24):
            for is_weekend in [0, 1]:
                for weather in ["Normal", "Severe", "Low Visibility"]:
                    # Base volume pattern with morning/evening peaks
                    base_vol = 1500 + 3500 * (
                        np.exp(-((hour - 8) ** 2) / 8)
                        + np.exp(-((hour - 17) ** 2) / 8)
                    )
                    if is_weekend:
                        base_vol *= 0.65
                    if weather == "Severe":
                        base_vol *= 0.85

                    vol = int(max(500, base_vol + np.random.normal(0, 200)))
                    risk = 0.05 + (0.25 if weather != "Normal" else 0.0) + (0.1 if 7 <= hour <= 9 or 16 <= hour <= 18 else 0.0)

                    records.append(
                        {
                            "hour": hour,
                            "is_weekend": is_weekend,
                            "weather_group": weather,
                            "traffic_volume": vol,
                            "high_risk": min(1.0, risk),
                        }
                    )
        return pd.DataFrame(records)

    def _build_lookup_profile(self):
        """Aggregates traffic volume and collision risk metrics into a multi-index lookup table."""
        if "day_type" not in self.df.columns:
            self.df["day_type"] = np.where(
                self.df["is_weekend"] == 1, "Weekend", "Weekday"
            )

        if "weather_group" not in self.df.columns:

            def classify_weather(row):
                if row.get("is_severe_weather", 0) == 1:
                    return "Severe"
                elif row.get("is_low_visibility", 0) == 1:
                    return "Low Visibility"
                return "Normal"

            self.df["weather_group"] = self.df.apply(classify_weather, axis=1)

        # Build aggregated profile
        self.profile = (
            self.df.groupby(["day_type", "weather_group", "hour"])
            .agg(
                avg_volume=("traffic_volume", "mean"),
                peak_volume=("traffic_volume", "max"),
                risk_rate=("high_risk", "mean"),
            )
            .reset_index()
        )

    def recommend_travel_window(
        self,
        day_type="Weekday",
        weather="Normal",
        search_window=(6, 20),
        duration_hours=1,
    ):
        """Recommends optimal travel departure windows based on parameters.

        Parameters:
        -----------
        day_type : str ('Weekday' or 'Weekend')
        weather : str ('Normal', 'Severe', 'Low Visibility')
        search_window : tuple (start_hour, end_hour) in 24h format (e.g., (6, 20))
        duration_hours : int (length of journey window)

        Returns:
        --------
        dict containing optimal window, expected volume, traffic savings, and plain-language narrative.
        """
        start_h, end_h = search_window

        # Filter profile matching parameters
        subset = self.profile[
            (self.profile["day_type"].str.lower() == day_type.lower())
            & (self.profile["weather_group"].str.lower() == weather.lower())
            & (self.profile["hour"] >= start_h)
            & (self.profile["hour"] <= end_h - duration_hours)
        ].copy()

        # Fallback to normal weather if specific condition combination is empty
        if subset.empty:
            subset = self.profile[
                (self.profile["day_type"].str.lower() == day_type.lower())
                & (self.profile["weather_group"] == "Normal")
                & (self.profile["hour"] >= start_h)
                & (self.profile["hour"] <= end_h - duration_hours)
            ].copy()

        # Calculate composite congestion & risk score
        subset["score"] = subset["avg_volume"] + (subset["risk_rate"] * 4000)

        # Identify optimal (lowest score) and worst (highest score) windows
        best_row = subset.sort_values("score").iloc[0]
        worst_row = subset.sort_values("score", ascending=False).iloc[0]

        best_hour = int(best_row["hour"])
        avg_vol = int(best_row["avg_volume"])
        peak_vol = int(worst_row["avg_volume"])
        volume_savings_pct = int(((peak_vol - avg_vol) / peak_vol) * 100)

        # Format 12-hour time strings
        window_start_str = (
            f"{best_hour % 12 or 12}:00 {'AM' if best_hour < 12 else 'PM'}"
        )
        end_hour_calc = best_hour + duration_hours
        window_end_str = f"{end_hour_calc % 12 or 12}:00 {'AM' if end_hour_calc < 12 else 'PM'}"

        # Build plain-language text recommendation
        weather_narrative = (
            f" under {weather.lower()} weather conditions"
            if weather.lower() != "normal"
            else ""
        )
        recommendation_text = (
            f"For a {day_type.lower()} journey{weather_narrative}, consider travelling between "
            f"{window_start_str} and {window_end_str}. Historical traffic volumes during this "
            f"window average ~{avg_vol:,} vehicles/hour—offering a {volume_savings_pct}% reduction "
            f"in traffic volume compared to peak travel periods (~{peak_vol:,} vehicles/hour)."
        )

        if best_row["risk_rate"] > 0.15:
            recommendation_text += (
                " Note: Weather conditions indicate elevated risk on this corridor; "
                "allow extra distance and reduce travel speed."
            )

        return {
            "day_type": day_type,
            "weather": weather,
            "recommended_window": f"{window_start_str} - {window_end_str}",
            "expected_avg_volume": avg_vol,
            "peak_period_volume": peak_vol,
            "volume_reduction": f"{volume_savings_pct}%",
            "plain_language_recommendation": recommendation_text,
        }


# Instantiate System
recommender = TravelTimingRecommender()

In [4]:
scenario_1 = recommender.recommend_travel_window(
    day_type="Weekday", weather="Normal", search_window=(7, 19), duration_hours=1
)
print("Plain Language Output:")
print(scenario_1["plain_language_recommendation"])

Plain Language Output:
For a weekday journey, consider travelling between 10:00 AM and 11:00 AM. Historical traffic volumes during this window average ~4,423 vehicles/hour—offering a 29% reduction in traffic volume compared to peak travel periods (~6,264 vehicles/hour).


In [5]:
scenario_2 = recommender.recommend_travel_window(
    day_type="Weekday", weather="Severe", search_window=(6, 18), duration_hours=1
)
print("Plain Language Output:")
print(scenario_2["plain_language_recommendation"])

Plain Language Output:
For a weekday journey under severe weather conditions, consider travelling between 10:00 AM and 11:00 AM. Historical traffic volumes during this window average ~4,170 vehicles/hour—offering a 26% reduction in traffic volume compared to peak travel periods (~5,655 vehicles/hour). Note: Weather conditions indicate elevated risk on this corridor; allow extra distance and reduce travel speed.


In [6]:
scenario_3 = recommender.recommend_travel_window(
    day_type="Weekend", weather="Normal", search_window=(8, 20), duration_hours=1
)
print("Plain Language Output:")
print(scenario_3["plain_language_recommendation"])

Plain Language Output:
For a weekend journey, consider travelling between 8:00 AM and 9:00 AM. Historical traffic volumes during this window average ~2,382 vehicles/hour—offering a 46% reduction in traffic volume compared to peak travel periods (~4,423 vehicles/hour).
